# Econ 521 · Lab Assignment: RD and DiD on Two Published Papers
### Academic probation (Lindo, Sanders & Oreopoulos 2010) and unilateral divorce (Wolfers 2006)


In the labs we built regression discontinuity (Labs 5–6) and difference-in-differences (Labs 7–8) from the
ground up, mostly on one running example each. This assignment hands you two published papers and asks
you to do what a careful referee does: **replicate the headline number, then push on it** until you know
which parts are solid and which depend on a choice someone made.

Both papers were chosen because each contains a trap that the modern toolkit exposes:

| | Part A · RD | Part B · DiD |
|---|---|---|
| **Paper** | Lindo, Sanders & Oreopoulos (2010), *AEJ: Applied* 2(2) | Wolfers (2006), *AER* 96(5), revisiting Friedberg (1998), *AER* 88(3) |
| **Question** | Does being put on academic probation make students work harder, or quit? | Did unilateral divorce laws raise divorce rates? |
| **Design** | Sharp RD at a GPA cutoff | Staggered adoption across US states, 1969–1985 |
| **The trap** | An outcome that only exists for students who stayed; a discrete running variable; subgroup results | A static coefficient plus state-specific trends, applied to an effect that rises and then fades |

**By the end you should be able to**
1. Separate what an estimate *is* from what a specification *does to it*: bandwidth, kernel, clustering, trends, weights.
2. Diagnose a design with the falsification toolkit, and read a failed diagnostic correctly instead of mechanically.
3. Handle post-treatment selection in an RD with bounds instead of pretending it isn't there.
4. Explain, with a simulation and with real data, why unit-specific trends can manufacture a treatment effect.
5. Write the honest one-sentence summary of a result, with the estimand stated.

### Instructions

* **Data.** Download `data_for_analysis.dta` (Lindo et al.) and `Divorce-Wolfers-AER.dta` (Wolfers) from Canvas
  and put them in a folder called `data/` next to this notebook.
* **Environment.** Python ≥ 3.11 with `rdrobust`, `rddensity`, `pyfixest==0.60.0`, `moderndid==0.2.0` (the same
  versions as the labs). On Colab, uncomment the `pip install` line in the setup cell.
* **What to fill in.** Code cells marked `# TODO` and markdown cells marked **Your answer**. Written answers should
  be short and specific: two to six sentences per item, **with numbers** from your output.
* **Checks.** Where a *published* number exists, `check()` tells you whether you replicated it. Everything
  else is yours to interpret; there is no answer printed for you.
* **Submit** the executed notebook (`.ipynb`) on Canvas. Every cell should run top to bottom.
* **Readings.** The two papers (posted). Optional: Friedberg (1998); Lee (2009) on bounds;
  Kolesár & Rothe (2018) on discrete running variables.

| Part | Content | Points |
|---|---|---|
| A | Regression discontinuity: academic probation | 45 |
| B | Difference-in-differences: unilateral divorce | 45 |
| C | Synthesis | 10 |

---
## 0 · Setup

In [ ]:
# On Colab or a fresh environment, uncomment the next line (needs Python >= 3.11):
# !pip install -q rdrobust rddensity pyfixest==0.60.0 moderndid==0.2.0

import io
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import pyfixest as pf
import moderndid as md
from rdrobust import rdrobust
from rddensity import rddensity

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.3f}".format

# Same palette as the labs. Blue = treated side / treated group, orange = comparison.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.figsize": (8, 4.2), "figure.dpi": 110,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 2, "legend.frameon": False, "legend.labelcolor": INK2,
})


def check(label, value, expected, tol=0.005):
    # Prints whether your number matches a PUBLISHED value (never raises)
    if value is None:
        print(f"[ ] {label}: not answered yet")
        return
    ok = abs(float(value) - expected) <= tol
    print(f"[{'✓' if ok else '✗'}] {label}: {float(value):,.3f}" + ("" if ok else "  ← not quite; re-check"))


def find_data(name):
    for p in [Path("data") / name, Path(name)]:
        if p.exists():
            return p
    raise FileNotFoundError(f"Put {name} in a folder called data/ next to this notebook (Canvas → Files).")

---
# Part A · Does academic probation motivate or discourage? (45 points)

> Lindo, J. M., Sanders, N. J., & Oreopoulos, P. (2010). "Ability, Gender, and Performance Standards:
> Evidence from Academic Probation." *American Economic Journal: Applied Economics* 2(2), 95–117.

**The institution.** At a large Canadian university with three campuses, every student is evaluated at the end
of the first year. A student whose cumulative GPA is **below** a cutoff is placed on **academic probation**:
the cutoff is 1.5 at Campuses 1 and 2 and 1.6 at Campus 3. The student gets a letter stating the standing, why,
how to regain good standing, and the consequences. A student on probation who does not bring the GPA back
above the cutoff at the next evaluation is **suspended** for a year.

**The question.** A performance standard can push two ways.
* **Motivation:** students just below the line work harder to avoid suspension.
* **Discouragement:** being told you are failing is information; some students update downward about their
  ability, or about whether university is worth it, and leave.

Both can be true for different students. The paper asks which dominates, and for whom.

**Why an RD is credible here.** First-year GPA is a credit-weighted average over many courses graded by
different instructors. To land on a chosen side of 1.50, a student would need to know the cutoff and control
their grades to the hundredth of a point. Only 15% of students surveyed by the authors said they knew the
cutoff "for sure."

**One convention to keep straight.** The running variable is $X_i = \text{GPA}_i - \text{cutoff}_{c(i)}$, and the
treatment switches on **below** the cutoff: $D_i = \mathbf{1}\{X_i < 0\}$. A student at exactly the cutoff is in
good standing.

### A0 · Data and helpers (no points; run these cells)

| Variable | Meaning |
|---|---|
| `x` (from `dist_from_cut`) | **Running variable**: first-year GPA minus the student's campus cutoff, on a 0.01 grid |
| `below` | 1 if `x < 0` (the probation side) |
| `probation_year1` | 1 if actually placed on probation after the first year |
| `left_school` | 1 if the student left the university after the first evaluation |
| `nextGPA` | GPA in the next term, **also measured as distance from the cutoff**. Missing if no next-term grades |
| `returned` | 1 if `nextGPA` is observed (we build it below) |
| `gradin4`, `gradin5`, `gradin6` | Graduated within 4 / 5 / 6 years (missing for cohorts not yet observed that long) |
| `hsgrade_pct` | High-school grade percentile (within the university) |
| `totcredits_year1` | Credits attempted in the first year |
| `age_at_entry`, `male`, `english`, `bpl_north_america` | Age, sex, English as first language, born in North America |
| `loc_campus1/2/3` | Campus indicators |
| `lowHS`, `highHS` | High-school grade percentile below / above the median |

In [ ]:
lso = pd.read_stata(find_data("data_for_analysis.dta"))
lso["x"] = lso["dist_from_cut"].astype(float).round(2)     # stored as float32; GPA lives on a 0.01 grid
lso["below"] = (lso["x"] < 0).astype(int)                  # 1 = probation side of the cutoff
lso["returned"] = lso["nextGPA"].notna().astype(int)        # 1 = we observe a next-term GPA
lso["campus"] = np.select([lso["loc_campus1"] == 1, lso["loc_campus2"] == 1], [1, 2], 3)

print(f"{len(lso):,} students entering {lso.firstyear.min()}–{lso.firstyear.max()}; "
      f"{(lso.x.abs() < 0.6).sum():,} within 0.6 grade points of their cutoff")
print("cutoff by campus:", lso.groupby("campus")["gpacutoff"].first().round(2).to_dict())
lso[["x", "below", "probation_year1", "left_school", "returned", "nextGPA", "hsgrade_pct", "male", "english"]].describe().T[["count", "mean", "std", "min", "max"]]

In [ ]:
def rd_probation(y, x, c=0.0, **kwargs):
    """Local-polynomial RD with rdrobust, reported as the effect of being BELOW the cutoff.

    rdrobust always reports  lim_{x↓c} E[Y|X=x] − lim_{x↑c} E[Y|X=x]   (right minus left).
    Probation switches on to the LEFT, so the effect of probation is minus that number and the CI flips.
    (A fuzzy estimate is a ratio of two right-minus-left jumps, so its sign needs no flip.)
    Defaults follow Lab 6: MSE-optimal bandwidth, triangular kernel, local linear, HC1 variance.
    Returns the point estimate, its conventional SE, and the ROBUST bias-corrected CI and p-value.
    """
    y = pd.Series(np.asarray(y, float)); x = pd.Series(np.asarray(x, float))
    ok = (y.notna() & x.notna()).to_numpy()
    for k in ("fuzzy", "cluster"):                    # keep optional vectors aligned with y and x
        if k in kwargs and kwargs[k] is not None:
            kwargs[k] = np.asarray(kwargs[k], float)[ok]
    kwargs.setdefault("vce", "hc1")
    with redirect_stdout(io.StringIO()):              # silences "Mass points detected in the running variable."
        o = rdrobust(y=y[ok].to_numpy(), x=x[ok].to_numpy(), c=c, **kwargs)
    sign = 1.0 if kwargs.get("fuzzy") is not None else -1.0
    lo, hi = sorted([sign * o.ci.iloc[2, 0], sign * o.ci.iloc[2, 1]])
    return pd.Series({"effect": sign * o.coef.iloc[0, 0], "se": o.se.iloc[0, 0], "p": o.pv.iloc[2, 0],
                      "CI lo": lo, "CI hi": hi, "h": o.bws.loc["h", "left"], "N": int(sum(o.N_h))})


def rd_paper(df, y, h=0.6, cluster=True):
    """The paper's specification: OLS on |x| < h with separate slopes on each side (a uniform kernel),
    SEs clustered on the values of the running variable (Lee & Card 2008). Same output format,
    but here p and the CI are conventional."""
    s = df.loc[df["x"].abs() < h].dropna(subset=[y])
    m = smf.ols(f"{y} ~ below + x + below:x", data=s)
    fit = m.fit(cov_type="cluster", cov_kwds={"groups": s["x"]}) if cluster else m.fit(cov_type="HC1")
    lo, hi = fit.conf_int().loc["below"]
    return pd.Series({"effect": fit.params["below"], "se": fit.bse["below"], "p": fit.pvalues["below"],
                      "CI lo": lo, "CI hi": hi, "h": h, "N": int(fit.nobs)})


def tidy(df):
    """Show sample sizes as integers in result tables."""
    return df.astype({"N": int}) if "N" in df.columns else df


def rd_plot(ax, df, y, ylabel, window=1.2, bin_width=0.1, h=0.6):
    """Binned means (0.1-wide bins, as in the paper's figures) plus a linear fit on each side within h."""
    s = df.loc[df["x"].abs() <= window].dropna(subset=[y])
    edges = np.round(np.arange(-window, window + bin_width / 2, bin_width), 10)
    b = (s.groupby(pd.cut(s["x"], edges, right=False), observed=True)
          .agg(xm=("x", "mean"), ym=(y, "mean")))
    left = b["xm"] < 0
    ax.axvspan(-window, 0, color=BLUE, alpha=0.06, lw=0)
    ax.scatter(b.loc[left, "xm"], b.loc[left, "ym"], color=BLUE, s=32, zorder=3, label="below: probation")
    ax.scatter(b.loc[~left, "xm"], b.loc[~left, "ym"], color=ORANGE, marker="^", s=32, zorder=3,
               label="at/above: good standing")
    for side, col in [(s["x"] < 0, BLUE), (s["x"] >= 0, ORANGE)]:
        part = s.loc[side & (s["x"].abs() < h)]
        slope, icept = np.polyfit(part["x"], part[y], 1)
        xs = np.linspace(part["x"].min(), part["x"].max(), 50)
        ax.plot(xs, icept + slope * xs, color=col, lw=2.2)
    ax.axvline(0, color=AXIS, lw=1, ls="--")
    ax.set_xlabel("first-year GPA − campus cutoff")
    ax.set_ylabel(ylabel)

---
### A1 · The rule, and what exactly crosses it (4 points)

**(a)** Compute the share of students placed on probation (`probation_year1`) below and at/above the cutoff,
separately by campus. **(b)** Estimate the jump in the probability of probation at the cutoff with
`rd_probation()`. **(c)** Estimate the effect on `left_school` twice: as a sharp RD (the effect of being below the
cutoff) and as a fuzzy RD with `fuzzy=lso["probation_year1"]` (the effect of probation itself).

In [ ]:
# TODO A1(a): share on probation by campus and side of the cutoff
#   hint: lso.groupby(["campus", "below"])["probation_year1"].mean().unstack("below")
first_stage_table = None

# TODO A1(b): the jump in P(probation_year1 = 1) at the cutoff, with rd_probation()
fs = None

# TODO A1(c): sharp and fuzzy estimates of the effect on left_school
sharp, fuzzy = None, None

if first_stage_table is not None:
    display(first_stage_table)
if fs is not None:
    display(tidy(fs.to_frame("jump in P(probation)").T))
if sharp is not None and fuzzy is not None:
    display(tidy(pd.DataFrame({"sharp: effect of being below the cutoff": sharp,
                               "fuzzy: effect of probation (Wald ratio)": fuzzy}).T))

Answer in the box below:
1. Is this a sharp or a fuzzy design? Use (a), (b) and (c). Contrast it with the exam-school
   simulation in Lab 6.
2. What exactly is the treatment? List what changes for a student who finishes the year at 1.49 rather than
   1.50, and say why it matters for how we interpret the estimate.
3. We pool three campuses by centering each student's GPA on their own campus cutoff. What parameter does the
   pooled RD estimate?
4. `rd_probation()` flips the sign of rdrobust's estimate. Why, and what would go wrong if you forgot?

**Your answer (A1):**

1. 
2. 
3. 
4. 

---
### A2 · Look before you estimate (3 points)

Draw the RD picture for three variables: `probation_year1` (the first stage), `left_school`, and `nextGPA`.
`rd_plot()` is defined in A0.

In [ ]:
# TODO A2: one panel per variable. Example: rd_plot(axes[0], lso, "probation_year1", "P(on probation)")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

axes[0].set_title("First stage"); axes[1].set_title("Leaving"); axes[2].set_title("Next-term GPA")
plt.tight_layout(); plt.show()

1. **First stage:** what does it look like, and what does that tell you about the design?
2. **Leaving:** the base rate near the cutoff, the slope, and whether you can see a jump by eye.
3. **Next-term GPA:** the slope and the jump. Is anything about this outcome different from the other two?

**Your answer (A2):**

1. 
2. 
3. 

---
### A3 · Replicate the headline, then modernize it (6 points)

**(a)** Run the paper's specification, `rd_paper()`, for `left_school` and `nextGPA`. The paper reports
**0.018** (Table 4) and **0.233** (Table 5). **(b)** Run the Lab 6 version, `rd_probation()`, for the same two
outcomes. **(c)** For `left_school` only, change *just the standard errors* in each: the paper's specification with
heteroskedasticity-robust SEs (`cluster=False`), and `rd_probation()` with SEs clustered on GPA values
(`cluster=lso["x"]`). The cell after yours lines up the four versions of the leaving estimate.

In [ ]:
# TODO A3(a): the paper's specification
paper_left = None
paper_gpa = None

# TODO A3(b): rdrobust with Lab 6 defaults
mod_left = None
mod_gpa = None

# TODO A3(c): same two estimators for left_school, changing only the standard errors
paper_left_hc1 = None
mod_left_cluster = None

check("Leaving, paper spec (Table 4: 0.018)", None if paper_left is None else paper_left["effect"], 0.018, tol=0.001)
check("Next-term GPA, paper spec (Table 5: 0.233)", None if paper_gpa is None else paper_gpa["effect"], 0.233, tol=0.001)
if all(v is not None for v in [paper_left, mod_left, paper_gpa, mod_gpa]):
    display(tidy(pd.DataFrame({"left_school (paper)": paper_left, "left_school (rdrobust)": mod_left,
                               "nextGPA (paper)": paper_gpa, "nextGPA (rdrobust)": mod_gpa}).T))

In [ ]:
# Four versions of the same estimate: what moves when you change the specification, and what moves with only the SEs?
versions = {"1. paper: h=0.6, uniform, SEs clustered on GPA": paper_left,
            "2. paper spec, heteroskedasticity-robust SEs": paper_left_hc1,
            "3. rdrobust: MSE-optimal h, triangular, robust": mod_left,
            "4. rdrobust, SEs clustered on GPA": mod_left_cluster}
if all(v is not None for v in versions.values()):
    spec_compare = pd.DataFrame(versions).T
    spec_compare["N"] = spec_compare["N"].astype(int)
    display(spec_compare)
else:
    print("Complete A3(a)-(c) to see the comparison.")

Rows 1–2 report conventional p-values and CIs; rows 3–4 report robust bias-corrected ones.

1. Report the paper's and the rdrobust estimates for both outcomes. Which conclusion survives the move to
   modern inference, and which does not?
2. Three things differ between rows 1 and 3. Name them. Using the table, which one is responsible for the change
   in the dropout p-value? (Compare the *point estimates* first.)
3. The paper clusters on the values of GPA, following Lee & Card (2008). Compare the standard errors in rows 1
   and 2, and the p-values in rows 3 and 4. Kolesár & Rothe (2018) show that clustering on a discrete running
   variable can produce confidence intervals that cover *less* often than plain heteroskedasticity-robust ones.
   Which number would you report, and what is the honest sentence about the dropout effect?

**Your answer (A3):**

1. 
2. 
3. 

---
### A4 · Can students sort across the line? (6 points)

**(a)** Run the manipulation test (`rddensity`) on `x` at 0. The histogram is drawn for you. **(b)** Build a
balance table: run `rd_probation()` with each predetermined covariate in `BAL_COVS` as the outcome, one row per
covariate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
near = lso.loc[lso["x"].abs() <= 0.6, "x"]
counts = near.value_counts().sort_index()
axes[0].bar(counts.index, counts.to_numpy(), width=0.008,
            color=np.where(counts.index < 0, BLUE, ORANGE))
axes[0].set_title("Students at each GPA value (0.01 grid)")
axes[1].hist(near, bins=np.round(np.arange(-0.6, 0.6001, 0.05), 3), color=MUTED, edgecolor=SURFACE)
axes[1].set_title("Same data, 0.05-wide bins")
for a in axes:
    a.axvline(0, color=INK, lw=1, ls="--"); a.set_xlabel("first-year GPA − cutoff")
axes[0].set_ylabel("students")
plt.tight_layout(); plt.show()

In [ ]:
# TODO A4(a): density (manipulation) test at the cutoff
dens = None
if dens is not None:
    print(f"density test: t = {dens.test['t_jk']:+.3f}, p = {dens.test['p_jk']:.3f}")

# TODO A4(b): balance table, one row per covariate (hint: a dict comprehension + pd.DataFrame(...).T)
BAL_COVS = ["hsgrade_pct", "totcredits_year1", "age_at_entry", "male",
            "bpl_north_america", "english", "loc_campus1", "loc_campus2"]
balance = None
if balance is not None:
    display(balance)

1. What does the density test say, and what does the left-hand histogram show about the running variable?
   Combine the test with the institutional argument.
2. Read the balance table. With eight covariates tested at 5%, what is the chance of at least one false
   rejection if all eight nulls are true and the tests were independent? Does any rejection survive a Bonferroni
   correction?
3. Suppose a covariate does jump. Give one concrete story of *student behaviour* that would produce a jump in
   `totcredits_year1` at the cutoff, and say what else you would expect to see in the data if that story were
   true.

**Your answer (A4):**

1. 
2. 
3. 

---
### A5 · One mass point, exactly at the cutoff (4 points)

The table below shows, for each GPA value within 0.05 of the cutoff, how many students sit there, their mean
credits attempted, and the share with exactly 4.0 credits.

In [ ]:
cells = (lso.loc[lso["x"].abs() <= 0.05]
            .groupby("x")
            .agg(students=("x", "size"),
                 mean_credits=("totcredits_year1", "mean"),
                 share_4_credits=("totcredits_year1", lambda v: (v == 4).mean()),
                 left_school=("left_school", "mean")))
cells.style.format({"mean_credits": "{:.2f}", "share_4_credits": "{:.0%}", "left_school": "{:.3f}"})

**Do this:** drop the students sitting **exactly** at the cutoff (`x == 0`) and re-run `rd_probation()` for
`totcredits_year1`, `left_school` and `nextGPA`. This is a donut with a hole of one GPA value.

In [ ]:
# TODO A5: donut RD that drops x == 0; one row per outcome
donut = None
if donut is not None:
    display(donut)

1. Why does $x = 0$ hold so many students, and why are so many of them on exactly 4.0 credits? (Hint: GPA is
   total grade points divided by credits.)
2. Explain how this one mass point produces a "jump" in credits in a local linear regression, and why that jump is
   not evidence of sorting.
3. Does the donut change the main results? State the general lesson about balance tests with a discrete running
   variable.

**Your answer (A5):**

1. 
2. 
3. 

---
### A6 · Bandwidth sensitivity and placebo cutoffs (5 points)

**(a)** For `left_school` and `nextGPA`, estimate `rd_probation(..., h=h)` on the grid `H_GRID` and store the results
in `path_left` and `path_gpa` (DataFrames with one row per bandwidth). The plot is drawn for you; it shows
conventional 95% intervals ($\hat\tau \pm 1.96\,\widehat{se}$), because rdrobust's robust intervals with a hand-picked
$h$ are not centered on $\hat\tau(h)$.

**(b)** Run placebo cutoffs for `left_school`, **one side at a time** (Lab 6, Section 2.3): at $-0.6, -0.4, -0.2$ using
only students below the real cutoff, and at $0.2, 0.4, 0.6, 0.8$ using only students at or above it.

In [ ]:
H_GRID = [0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2]

# TODO A6(a): bandwidth paths (hint: pd.DataFrame([rd_probation(..., h=h) for h in H_GRID]))
path_left = None
path_gpa = None

# TODO A6(b): placebo cutoffs for left_school, one side at a time; one row per placebo cutoff
placebo = None

if path_left is not None and path_gpa is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, path, opt, title in [(axes[0], path_left, mod_left, "Leaving"), (axes[1], path_gpa, mod_gpa, "Next-term GPA")]:
        ax.fill_between(path["h"], path["effect"] - 1.96 * path["se"], path["effect"] + 1.96 * path["se"],
                        color=BLUE, alpha=0.15, lw=0, label="conventional 95% CI")
        ax.plot(path["h"], path["effect"], "o-", color=BLUE, label=r"$\hat\tau(h)$")
        ax.axhline(0, color=INK, lw=1)
        if opt is not None:
            ax.axvline(opt["h"], color=ORANGE, ls=":", lw=2, label=f"MSE-optimal h = {opt['h']:.2f}")
        ax.set_xlabel("bandwidth h"); ax.set_title(title); ax.legend(fontsize=8, loc="best")
    plt.tight_layout(); plt.show()
if placebo is not None:
    display(tidy(placebo))

1. For each outcome: is $\hat\tau(h)$ a plateau or a drift? At which bandwidths does the conventional interval for
   leaving exclude zero?
2. What do the placebo cutoffs say?
3. Write the honest one-sentence summary of the bandwidth evidence for **leaving**, in the style of Lab 6's Hansen
   write-up.

**Your answer (A6):**

1. 
2. 
3. 

---
### A7 · The outcome only exists for students who came back (8 points)

`nextGPA` is observed only for students who take courses next term. Probation changes who does. So the RD on
next-term GPA compares **returners below** the cutoff with **returners above** it, and if probation pushes some
students out, those two groups are no longer comparable, even at the cutoff. The RD's "as good as random at
the cutoff" property holds for the *full* population of students; selection that happens *after* treatment
undoes it.

**Lee (2009) bounds** handle this without modelling who leaves. The idea, with one assumption:

* **Monotonicity:** probation can make a student *not* return, but never makes someone return who otherwise
  would not have. Then returners below the cutoff are "always-returners," while returners above the cutoff are
  always-returners **plus** a share $p$ of students who would have left had they been put on probation.
* We don't know *which* above-cutoff returners are the extra ones. **Worst case for "probation helps":** they were
  the weakest, so drop the bottom $p$ share of above-cutoff outcomes (the comparison mean rises, giving the
  **lower** bound). **Best case:** they were the strongest, so drop the top $p$ share (giving the **upper** bound).
* With $q_0$ and $q_1$ the return rates just above and just below the cutoff: $p = (q_0 - q_1)/q_0$.

We work in the paper's window ($|x| < 0.6$) and trim relative to the fitted line on the above side, so that we
compare students with the same first-year GPA. The trimming helper is given.

In [ ]:
W6 = lso.loc[lso["x"].abs() < 0.6].copy()


def lee_bound(p, side):
    """Trim a share p of the ABOVE-cutoff returners (by their residual from a linear fit on x) and re-run
    the paper's RD on next-term GPA. side='bottom' gives the lower bound, side='top' the upper bound."""
    ret = W6.dropna(subset=["nextGPA"])
    above, below_ = ret.loc[ret["below"] == 0], ret.loc[ret["below"] == 1]
    resid = smf.ols("nextGPA ~ x", data=above).fit().resid
    keep = resid >= resid.quantile(p) if side == "bottom" else resid <= resid.quantile(1 - p)
    return rd_paper(pd.concat([below_, above.loc[keep]]), "nextGPA")["effect"]

**(a)** Estimate the effect of probation on `returned` with both `rd_paper()` and `rd_probation()`, and compare it
with the effect on `left_school`. **(b)** From the paper-spec regression of `returned` on `below + x + below:x`
(on `W6`), recover $q_0$ (the intercept) and $q_1$ ($q_0$ + the coefficient on `below`), compute $p$, and call
`lee_bound()` for both bounds.

In [ ]:
# TODO A7(a): effect of probation on returning (next-term GPA observed), two ways
ret_paper = None
ret_mod = None

# TODO A7(b): return rates at the cutoff, trimming share, and the two bounds
q0, q1, p_trim = None, None, None
lower, upper = None, None

if p_trim is not None:
    print(f"q0 = {q0:.3f}, q1 = {q1:.3f}, trimming share p = {p_trim:.3f}")
if lower is not None and upper is not None:
    print(f"Next-term GPA: point estimate {rd_paper(lso, 'nextGPA')['effect']:.3f}, Lee bounds [{lower:.3f}, {upper:.3f}]")

In [ ]:
# For item 4: the same bounds if we trim by the effect on formally LEAVING instead of on RETURNING
m_left = smf.ols("left_school ~ below + x + below:x", data=W6).fit()
p_left = m_left.params["below"] / (1 - m_left.params["Intercept"])
print(f"trimming share from left_school: p = {p_left:.3f}, "
      f"bounds [{lee_bound(p_left, 'bottom'):.3f}, {lee_bound(p_left, 'top'):.3f}]")

1. Why is the RD on next-term GPA not, by itself, a clean causal effect of probation? Who are the students that
   probation removes from the comparison?
2. Interpret your bounds. Does the motivation conclusion survive the worst case?
3. Is monotonicity plausible here? Describe a student who would violate it.
4. The paper reports bounds of $[0.186, 0.259]$. Yours are wider. The effect on `returned` is about twice the effect
   on `left_school`. Which selection indicator is the right one to trim on, and why? (The cell prints what you
   would get with the other one.)

**Your answer (A7):**

1. 
2. 
3. 
4. 

---
### A8 · Who is discouraged? Heterogeneity and the garden of forking paths (5 points)

The paper (Table 4 and its discussion) emphasizes that the dropout effect is concentrated among **men** and
**native English speakers**, and among students with strong high-school records.

**(a)** For each subgroup in `GROUPS`, estimate the effect on `left_school` with `rd_probation()`; one row per group.
**(b)** Test whether the effect differs between men and women, and between native and non-native English speakers:
in the paper's window `W6`, regress `left_school` on `(below + x + below:x) * male` (and `* english`) with
heteroskedasticity-robust SEs (`cov_type="HC1"`, following A3). The coefficient on `below:male` is the difference. **(c)** Apply a Holm correction to the six
subgroup p-values from (a) with `multipletests(pvals, method="holm")`.

In [ ]:
GROUPS = {"men": lso["male"] == 1, "women": lso["male"] == 0,
          "native English": lso["english"] == 1, "non-native English": lso["english"] == 0,
          "HS grades below median": lso["lowHS"] == 1, "HS grades above median": lso["highHS"] == 1}

# TODO A8(a): subgroup table (one rd_probation() row per group)
subgroups = None

# TODO A8(b): differences between subgroups, HC1 standard errors
diff_male, diff_english = None, None      # each a fitted statsmodels result

# TODO A8(c): Holm-adjusted p-values (add a column "Holm p" to subgroups)

if subgroups is not None:
    display(subgroups)
for name, fit, term in [("men − women", diff_male, "below:male"), ("native − non-native", diff_english, "below:english")]:
    if fit is not None:
        print(f"{name:20s}: difference {fit.params[term]:+.4f} (SE {fit.bse[term]:.4f}), p = {fit.pvalues[term]:.3f}")

1. Summarize the subgroup estimates. Which survive the Holm correction?
2. "The effect is significant for men and not for women, so probation discourages men more." What is wrong with
   this sentence, and what does your test in (b) say?
3. Count the researcher degrees of freedom in this part: outcomes × subgroups × bandwidths × SE choices. Connect
   this to the pre-analysis plans in Lab 3. What would you have written down *before* seeing the data?

**Your answer (A8):**

1. 
2. 
3. 

---
### A9 · What did we learn? (4 points)

The cell below estimates the effect on graduation and shows which entry cohorts each graduation outcome is
observed for. Use it for item 3.

In [ ]:
grad = pd.DataFrame({y: rd_probation(lso[y], lso["x"]) for y in ["gradin4", "gradin5", "gradin6"]}).T
grad["entry cohorts observed"] = [f"{int(lso.loc[lso[y].notna(), 'firstyear'].min())}–{int(lso.loc[lso[y].notna(), 'firstyear'].max())}"
                                  for y in grad.index]
grad["N"] = grad["N"].astype(int)
grad

1. State the estimand of the headline estimates in one sentence that a dean could understand: for whom, compared
   with what?
2. Motivation or discouragement? Give your verdict with the honest level of uncertainty from A3–A8.
3. Does probation change graduation? Why do the graduation samples shrink from 4 to 6 years, and what does that
   do to the comparison across the three columns?
4. The dean proposes raising the cutoff to 1.7. What can this RD tell her, and what can't it?

**Your answer (A9):**

1. 
2. 
3. 
4. 

---
# Part B · Did unilateral divorce laws raise divorce rates? (45 points)

> Wolfers, J. (2006). "Did Unilateral Divorce Laws Raise Divorce Rates? A Reconciliation and New Results."
> *American Economic Review* 96(5), 1802–1820.
>
> Friedberg, L. (1998). "Did Unilateral Divorce Raise Divorce Rates? Evidence from Panel Data."
> *American Economic Review* 88(3), 608–627.

**The institution.** Until the late 1960s, ending a marriage in most US states required proving fault (adultery,
cruelty, desertion) or getting the spouse's consent. Between 1969 and 1977 most states moved to **unilateral
divorce**: either spouse can end the marriage without the other's agreement. A few states had done so long
before, and about two in five never did during the sample period.

**Why economists care: a test of Coase.** Becker's argument: if spouses can bargain cheaply, the law changes *who
has to compensate whom*, not *whether* a marriage ends. Under mutual consent, a spouse who wants out pays the
other to agree; under unilateral divorce, a spouse who wants to stay pays the other to stay. Either way, marriages
end exactly when they are worth more apart than together. So the **Coasean prediction is no effect on divorce
rates.** A positive effect means bargaining inside marriage is costly or incomplete.

**The debate.** Friedberg (1998) used a state-year panel, 1968–1988, with state and year fixed effects and
**state-specific trends**, and found that unilateral divorce raised the divorce rate by about 0.45 divorces per
1,000 people, permanently. Wolfers (2006) argued the response is **dynamic**: a spike that fades within a
decade. He argued that a single "after reform" dummy combined with state trends mistakes that shape for a level
shift. Your job is to see why, and then check his answer with the tools from Lab 8.

### B0 · Data (no points; run these cells)

| Variable | Meaning |
|---|---|
| `st`, `year` | State (50 states + DC) and year, 1956–1998 |
| `div_rate` | **Outcome**: divorces per 1,000 people (some state-years missing) |
| `stpop` | State population |
| `lfdivlaw` | Year of reform, Friedberg's coding. Before 1956 = reformed before the sample starts; 2000 = never reformed by 1998 |
| `unilateral` | Friedberg's treatment dummy: 1 from the reform year on |
| `years_unilateral` | Wolfers' event-time bins: 1 = first two years of the regime (reform year and the next), 3 = years 3–4, …, 15 = 15+ years; −99 = before reform or never |
| `time`, `timesq` | Year − 1968 and its square (for state-specific trends) |
| `divx1`–`divx17` | Friedberg's dummies for breaks in individual states' divorce series |
| `gruber`, `johnson`, `mechoulan` | Reform year under other authors' codings (0 = no reform) |

In [ ]:
div = pd.read_stata(find_data("Divorce-Wolfers-AER.dta"))
div["year"] = div["year"].astype(int)
div["stpop"] = div["stpop"].astype(float)
print(f"{div.st.nunique()} states (incl. DC), {div.year.min()}–{div.year.max()}; "
      f"{div.div_rate.isna().sum()} state-years with a missing divorce rate")

# Timing group G (Friedberg's coding, used throughout):
#   G = reform year if the state reforms inside 1956–1988;  G = 0 if it never reforms;  G = -1 if it reformed before 1956.
div["G"] = np.select([div["lfdivlaw"].between(1956, 1988), div["lfdivlaw"] == 2000], [div["lfdivlaw"], 0], -1).astype(int)


def plot_event_study(ax, e, att, se, crit=None, color=BLUE, label=None, offset=0.0):
    # Dark bars: pointwise 95% CIs. Light bars: uniform (simultaneous) 95% band, if a critical value is given.
    e = np.asarray(e, float) + offset
    att, se = np.asarray(att, float), np.nan_to_num(np.asarray(se, float))
    if crit is not None:
        crit = np.broadcast_to(np.asarray(crit, float), att.shape)
        ax.vlines(e, att - crit * se, att + crit * se, color=color, alpha=0.3, lw=1.5)
    ax.vlines(e, att - 1.96 * se, att + 1.96 * se, color=color, lw=2.5)
    ax.plot(e, att, "o", color=color, ms=6, mec=SURFACE, mew=1.2, label=label, zorder=3)
    ax.axhline(0, color=AXIS, lw=1)


def es_coefs(fit, var="rel"):
    # Event-time coefficients from a pyfixest fit with i(var, ref=...): returns (event times, estimates, SEs)
    c, s = fit.coef(), fit.se()
    keep = [k for k in c.index if k.startswith(f"{var}::")]
    return np.array([int(k.split("::")[1]) for k in keep]), c[keep].to_numpy(), s[keep].to_numpy()

---
### B1 · Who is treated, and when? (3 points)

`G` is built for you above. **(a)** Make a table `cohorts` with one row per value of `G`: the number of states, their
names, and their share of the 1968 US population. **(b)** Look at the missing-data table printed below it. Then run the
next cell, which builds the **balanced panel** we use for the modern estimators in B2 and B6–B8.

In [ ]:
# TODO B1(a): one row per G — number of states, list of states, share of 1968 population
cohorts = None
if cohorts is not None:
    display(cohorts)

# Where are the holes? (given)
missing = div.loc[div["div_rate"].isna()].groupby("st")["year"].agg(["min", "max", "size"])
missing.columns = ["first missing year", "last missing year", "missing years (1956–1998)"]
display(missing)

In [ ]:
# Balanced panel for B2 and B6–B8: years 1959–1988, states with no missing divorce rate in that window,
# never-treated and in-sample adopters only (G >= 0). Starting in 1959 rather than 1956 keeps six states with gaps
# in 1956–58 (IL, KY, MA, NC, NY, WV; Illinois's other gap, 1991, is outside the window).
# Weights: 1968 population, held fixed over time.
# moderndid 0.2.0 only carries weights into aggte() if the column is literally called "weights" (Lab 8 gotcha).
window = div.loc[div["year"].between(1959, 1988)]
incomplete = sorted(window.loc[window["div_rate"].isna(), "st"].unique())
panel = window.loc[~window["st"].isin(incomplete) & (window["G"] >= 0)].copy()
panel["id"] = panel["st"].astype("category").cat.codes + 1
panel["weights"] = panel["st"].map(panel.loc[panel["year"] == 1968].set_index("st")["stpop"])
panel["D"] = ((panel["G"] > 0) & (panel["year"] >= panel["G"])).astype(int)

units = panel.drop_duplicates("st").set_index("st")
print(f"dropped for gaps in 1959–1988: {incomplete};  dropped as always treated: "
      f"{sorted(div.loc[div['G'] == -1, 'st'].unique())}")
print(f"balanced panel: {len(units)} states × {panel.year.nunique()} years; "
      f"{(units.G == 0).sum()} never reform, {(units.G > 0).sum()} reform in {sorted(units.loc[units.G > 0, 'G'].unique())}")

1. How many states never reform, how many reform inside the sample, and how many were already treated in 1956?
   Which cohort is largest? What share of the population lives in never-reforming states?
2. Alaska and Oklahoma reformed before 1956. In a TWFE regression on the full sample they stay in. What role do
   they play there, and why must Callaway–Sant'Anna drop them?
3. The never-reforming states include New York, Pennsylvania, Illinois, Ohio and much of the South. Give one reason
   to doubt that their divorce trends are a good counterfactual for the reformers.

**Your answer (B1):**

1. 
2. 
3. 

---
### B2 · One 2×2 to start (4 points)

The **1971 cohort** (Alabama, Colorado, Florida, Idaho, New Hampshire, North Dakota) against the **never-reforming**
states, comparing **1970** (the last pre-reform year) with **1972**. The picture is drawn for you.

In [ ]:
def wmean(g, col="div_rate"):
    return np.average(g[col], weights=g["weights"])

paths = (panel.loc[panel["G"].isin([0, 1971])]
              .assign(group=lambda d: np.where(d["G"] == 1971, "Reformed in 1971", "Never reformed"))
              .groupby(["group", "year"])[["div_rate", "weights"]].apply(wmean).unstack("group"))
fig, ax = plt.subplots()
for col, color, marker in [("Reformed in 1971", BLUE, "o"), ("Never reformed", ORANGE, "^")]:
    ax.plot(paths.index, paths[col], color=color, marker=marker, ms=5, mec=SURFACE, label=col)
ax.axvline(1970.5, color=AXIS, lw=1)
ax.set_ylabel("divorces per 1,000 people"); ax.set_title("Population-weighted divorce rates")
ax.legend(); plt.show()
print("1968 population of the 1971 cohort (millions):",
      (units.loc[units.G == 1971, "weights"] / 1e6).round(2).to_dict())

In [ ]:
# TODO B2: the 2x2 DiD, 1970 vs 1972, 1971 cohort vs never-reformers, using panel
#   (a) population-weighted (use the "weights" column), (b) unweighted
did_2x2_w = None
did_2x2_u = None
if did_2x2_w is not None:
    print(f"2x2 DiD, weighted: {did_2x2_w:+.3f}   unweighted: {did_2x2_u:+.3f}")

1. State, in one sentence each, the parameter this 2×2 estimates and the parallel-trends assumption it needs, in
   words about states and divorces.
2. The weighted and unweighted numbers differ a lot. What is going on? (Look at who is in the cohort.)
3. Look at the picture before 1971. Is parallel trends plausible for this 2×2? Does your answer depend on whether you
   think in levels or in percentage changes?

**Your answer (B2):**

1. 
2. 
3. 

---
### B3 · Friedberg's specification (4 points)

Wolfers' Table 1 replicates Friedberg on her sample, 1968–1988, weighting by state population. Column (1) is given.
Add **(2)** state-specific linear trends and **(3)** state-specific linear and quadratic trends. In `pyfixest`,
`i(st, time)` creates one slope on `time` per state. Wolfers reports **0.000**, **0.431** and **0.435**.

In [ ]:
fried = div.loc[div["year"].between(1968, 1988)].dropna(subset=["div_rate"]).copy()
DIVX = " + ".join(f"divx{i}" for i in range(1, 18))       # Friedberg's dummies for breaks in state series
f1 = pf.feols(f"div_rate ~ unilateral + {DIVX} | st + year", data=fried, weights="stpop", vcov={"CRV1": "st"})
print(f"N = {len(fried)};  (1) state + year FE: {f1.coef()['unilateral']:+.3f} (SE {f1.se()['unilateral']:.3f})")

In [ ]:
# TODO B3: add state-specific linear trends (f2), then linear + quadratic trends (f3). Same weights and clustering.
f2 = None
f3 = None

check("(2) linear state trends (Wolfers Table 1: 0.431)", None if f2 is None else f2.coef()["unilateral"], 0.431, tol=0.002)
check("(3) quadratic state trends (Wolfers Table 1: 0.435)", None if f3 is None else f3.coef()["unilateral"], 0.435, tol=0.002)

1. Adding state trends moves the estimate from zero to 0.43. Give **two** different stories that would each produce
   this pattern. (One should be Friedberg's.)
2. Can this table alone tell the two stories apart? What would you need to see?

**Your answer (B3):**

1. 
2. 

---
### B4 · Wolfers' event study (7 points)

Replace the single dummy with Wolfers' two-year event-time bins (`years_unilateral`) and extend the sample back to
1956, so trends can be estimated from pre-reform years. The bin dummies are built for you. Estimate **(1)** state
and year FE only and **(2)** with state-specific linear trends, both population-weighted with SEs clustered by
state. Then refit (1) with heteroskedasticity-robust SEs (`vcov="hetero"`), which is closer to what was standard in 2006.

In [ ]:
wolf = div.loc[div["year"].between(1956, 1988)].dropna(subset=["div_rate"]).copy()
BINS = [1, 3, 5, 7, 9, 11, 13, 15]
BIN_LABELS = ["1–2", "3–4", "5–6", "7–8", "9–10", "11–12", "13–14", "15+"]
for b in BINS:
    wolf[f"yu{b}"] = (wolf["years_unilateral"] == b).astype(float)
YU = " + ".join(f"yu{b}" for b in BINS)
print(f"N = {len(wolf)} state-years; population-weighted mean divorce rate {np.average(wolf.div_rate, weights=wolf.stpop):.2f} per 1,000")
print(f"formula terms: {YU}")

# For comparison: the static dummy on this longer sample
s1 = pf.feols("div_rate ~ unilateral | st + year", data=wolf, weights="stpop", vcov={"CRV1": "st"})
s2 = pf.feols("div_rate ~ unilateral + i(st, time) | st + year", data=wolf, weights="stpop", vcov={"CRV1": "st"})
print(f"static dummy, 1956–1988: no trends {s1.coef()['unilateral']:+.3f};  with state trends {s2.coef()['unilateral']:+.3f}")

In [ ]:
# TODO B4: Wolfers' dynamic specification
es_fe = None        # (1) state + year FE, clustered by state
es_trend = None     # (2) + state-specific linear trends, clustered by state
es_fe_hc = None     # (1) again, vcov="hetero"

if es_fe is not None and es_trend is not None:
    wolfers_table = pd.DataFrame({
        "(1) FE": es_fe.coef()[[f"yu{b}" for b in BINS]].to_numpy(),
        "SE clustered": es_fe.se()[[f"yu{b}" for b in BINS]].to_numpy(),
        "SE hetero": None if es_fe_hc is None else es_fe_hc.se()[[f"yu{b}" for b in BINS]].to_numpy(),
        "(2) + state trends": es_trend.coef()[[f"yu{b}" for b in BINS]].to_numpy(),
        "SE clustered ": es_trend.se()[[f"yu{b}" for b in BINS]].to_numpy(),
    }, index=pd.Index(BIN_LABELS, name="years since reform"))
    display(wolfers_table)

    fig, ax = plt.subplots()
    pos = np.arange(len(BINS))
    plot_event_study(ax, pos, wolfers_table["(1) FE"], wolfers_table["SE clustered"], color=BLUE, label="(1) state + year FE", offset=-0.1)
    plot_event_study(ax, pos, wolfers_table["(2) + state trends"], wolfers_table["SE clustered "], color=ORANGE, label="(2) + state linear trends", offset=0.1)
    ax.set_xticks(pos, BIN_LABELS); ax.set_xlabel("years since reform")
    ax.set_ylabel("divorces per 1,000 people"); ax.set_title("Wolfers (2006): the effect over time (95% CIs, clustered by state)")
    ax.legend(); plt.show()

1. Describe the path in column (1). How large is the initial effect relative to the mean divorce rate printed above?
2. On this longer sample, the static dummy is about zero without trends and about +0.48 with them, the same
   pattern as Friedberg's. Using the event-study table, explain precisely **why** the static dummy with state trends
   is large, even though no bin in column (1) is as large as 0.48.
3. In the dynamic specification, adding trends shifts the early bins less than the late ones. Look at how the shift
   grows with event time. Why are the later bins so sensitive to the trend specification?
4. Compare the clustered and heteroskedasticity-robust SEs. Why are they so different, and which do you report?

**Your answer (B4):**

1. 
2. 
3. 
4. 

---
### B5 · Simulate it: when a state trend eats the treatment effect (7 points)

Below is a fake state-year panel built to look like the divorce data, with one crucial feature: **there are no
state-specific trends at all**. Parallel trends holds exactly. The effect jumps to `spike` in the reform year and
then decays: $\tau(e) = \text{spike}\times 0.5^{\,e/\text{half-life}}$.

In [ ]:
def simulate_reform(spike=0.4, half_life=3.0, seed=521):
    """State-year panel 1956–1988: 24 states adopt between 1969 and 1977, 20 never do.
    Common upward trend in divorce, NO state-specific trends, effect that spikes and fades."""
    rng = np.random.default_rng(seed)
    adopt = [1969, 1970, 1971, 1971, 1972, 1973, 1973, 1973, 1974, 1975, 1976, 1977] * 2
    rows = []
    for i, g in enumerate(adopt + [0] * 20):
        level = rng.normal(4, 1)                                   # states differ in levels (fine under PT)
        for t in range(1956, 1989):
            e = t - g
            tau = spike * 0.5 ** (e / half_life) if (g > 0 and e >= 0) else 0.0
            rows.append((i, t, g, level + 0.08 * (t - 1956) + tau + rng.normal(0, 0.15), tau))
    d = pd.DataFrame(rows, columns=["id", "year", "G", "y", "tau"])
    d["D"] = ((d["G"] > 0) & (d["year"] >= d["G"])).astype(int)
    d["time"] = d["year"] - 1968
    d["rel"] = np.where(d["G"] > 0, (d["year"] - d["G"]).clip(-5, 15), -1)   # event time, binned at -5 and 15+
    return d


sim = simulate_reform()
print(f"{sim.id.nunique()} states, {sim.year.nunique()} years; effect in the reform year = 0.40, half-life = 3 years")

**(a)** Compute the true average effect over treated state-years (mean of `tau` where `D == 1`), the static TWFE
estimate (`y ~ D | id + year`), and the static TWFE with unit-specific linear trends (`+ i(id, time)`).
**(b)** Estimate the event study `y ~ i(rel, ref=-1) | id + year` with and without unit trends. The plot is drawn
for you.

In [ ]:
# TODO B5(a)
true_avg = None
b_static = None
b_static_trend = None
if b_static_trend is not None:
    print(f"true average effect {true_avg:.3f} | static TWFE {b_static:.3f} | static TWFE + unit trends {b_static_trend:.3f}")

# TODO B5(b): event studies (fitted pyfixest objects)
sim_es = None
sim_es_trend = None

if sim_es is not None and sim_es_trend is not None:
    fig, ax = plt.subplots()
    for fit, color, lab, off in [(sim_es, BLUE, "no unit trends", -0.12), (sim_es_trend, ORANGE, "with unit trends", 0.12)]:
        plot_event_study(ax, *es_coefs(fit), color=color, label=lab, offset=off)
    ee = np.arange(0, 16)
    ax.plot(ee, 0.4 * 0.5 ** (ee / 3.0), color=INK, ls="--", lw=1.2, label="true effect")
    ax.set_xlabel("event time"); ax.set_title("Simulated event studies"); ax.legend(); plt.show()

1. Compare the three numbers in (a). Which estimator is close to the truth, and which is badly off, in which
   direction?
2. Explain the bias using the DGP: there are no state trends, so what is the unit-specific trend picking up?
3. Why does the event study with unit trends get the path (nearly) right while the static model with unit trends
   fails? What does that tell you about *when* unit-specific trends are a legitimate tool?
4. Lab 8 showed static TWFE breaking with staggered adoption because of forbidden comparisons. Is that the main
   problem here, or is it the trends? Use your numbers.

**Your answer (B5):**

1. 
2. 
3. 
4. 

---
### B6 · Callaway & Sant'Anna on the divorce panel (9 points)

Now the modern estimator, on the balanced panel from B1 (`panel`: 46 states, 1959–1988).

**(a)** Estimate all $ATT(g,t)$ with `md.att_gt`: never-treated comparison group, universal base period, population
weights (`weightsname="weights"`), multiplier bootstrap with 999 draws and uniform bands, `random_state=521`.
**(b)** Aggregate to an event study for $e$ from −10 to 18 (`md.aggte(cs, type="dynamic", min_e=-10, max_e=18)`) and
store a DataFrame `es_df` with columns `e`, `att`, `se`, `crit` (from `.event_times`, `.att_by_event`, `.se_by_event`,
`.critical_values`, as in Lab 8). **(c)** For comparison, estimate a TWFE event study on the same panel, with leads and
lags: `div_rate ~ i(rel, ref=-1) | st + year`, population-weighted (`weights="weights"`), SEs clustered by state, where
`rel` (built for you) is event time binned at −10 and 15+, and never-treated states get −1. **(d)** Fill the
aggregation table: CS *simple*, *group*, dynamic averaged over $e \in [0,7]$, dynamic averaged over $e \in [8,15]$, and
the static TWFE (`div_rate ~ D | st + year`) with and without state trends on `panel`, weighted and clustered the
same way.

In [ ]:
panel["rel"] = np.where(panel["G"] > 0, (panel["year"] - panel["G"]).clip(-10, 15), -1)
print(panel.groupby("G")["rel"].agg(["min", "max"]).T)

In [ ]:
# TODO B6(a)
cs = None

# TODO B6(b): es_df with columns e, att, se, crit
es_df = None

# TODO B6(c): TWFE event study on panel (fitted pyfixest object)
twfe_es = None

if es_df is not None:
    show = es_df.loc[es_df["e"].between(-10, 18)]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    plot_event_study(ax, show["e"], show["att"], show["se"], crit=show["crit"], color=BLUE,
                     label="Callaway–Sant'Anna (never-treated; light bars = uniform band)", offset=-0.15)
    if twfe_es is not None:
        plot_event_study(ax, *es_coefs(twfe_es), color=ORANGE, label="TWFE event study (leads & lags)", offset=0.15)
    ax.axvline(-0.5, color=AXIS, lw=1)
    ax.set_xlabel("years since reform (e = 0 is the reform year)"); ax.set_ylabel("divorces per 1,000 people")
    ax.set_title("Unilateral divorce: event study on the balanced panel"); ax.legend(loc="lower left", fontsize=8)
    plt.show()

In [ ]:
# Sanity check: your B2 2x2 is one of the ATT(g, t) cells
if cs is not None and did_2x2_w is not None:
    gt = pd.DataFrame({"g": cs.groups.astype(int), "t": cs.times.astype(int), "att": cs.att_gt})
    cell_1971_1972 = gt.loc[(gt.g == 1971) & (gt.t == 1972), "att"].iloc[0]
    print(f"your 2x2 (B2): {did_2x2_w:+.4f}   att_gt's ATT(1971, 1972): {cell_1971_1972:+.4f}")

In [ ]:
# TODO B6(d): one row per summary; columns "estimate" and "SE"
agg_table = None
if agg_table is not None:
    display(agg_table)

In [ ]:
# For item 4: the event study using only cohorts observed at every event time 0..15 (reform in 1973 or earlier)
if cs is not None:
    bal = md.aggte(cs, type="dynamic", balance_e=15)
    print("balanced in event time:", pd.Series(bal.att_by_event, index=bal.event_times.astype(int)).loc[0:15].round(2).to_dict())

1. Does CS confirm Wolfers' story? Describe the pre-period estimates and the post-period path, and say which
   statements the **uniform** band lets you make.
2. How close are the TWFE event study and CS? Using Lab 8, explain why the forbidden-comparison problem is (or is
   not) serious in this application.
3. Report the *simple* and *group* overall ATTs. Why are they a misleading answer to "did unilateral divorce raise
   divorce rates?" Which summary would you report, and what do the CS aggregations say about the static TWFE with
   trends?
4. Which cohorts contribute to the estimates at $e = 15$ and beyond? What does that do to how you read the right end
   of the plot? Use the balanced-in-event-time output printed above.

**Your answer (B6):**

1. 
2. 
3. 
4. 

---
### B7 · Weights decide the question, and expose an outlier (5 points)

Re-estimate the CS event study (no bootstrap needed: `boot=False`) **(i)** unweighted, **(ii)** unweighted without
Nevada, and **(iii)** population-weighted without California. Store each as a DataFrame with columns `e` and `att`
in the dict `alt_paths`. The plot overlays them on your weighted estimate from B6.

In [ ]:
nv_ca = (panel.loc[panel["st"].isin(["NV", "CA"])].pivot(index="year", columns="st", values="div_rate")
              .loc[[1959, 1965, 1969, 1970, 1971, 1973, 1975, 1980, 1985, 1988]])
print("Divorce rate per 1,000 people (Nevada reformed in 1973, California in 1970):")
print(nv_ca.round(1).T)
print(f"\nshare of the panel's 1968 population: NV {units.loc['NV', 'weights'] / units.weights.sum():.2%}, "
      f"CA {units.loc['CA', 'weights'] / units.weights.sum():.1%}")

In [ ]:
# TODO B7: three alternative CS event studies; each value a DataFrame with columns e, att
alt_paths = {
    "unweighted": None,
    "unweighted, without Nevada": None,
    "weighted, without California": None,
}

fig, ax = plt.subplots(figsize=(10, 4.2))
if es_df is not None:
    s = es_df.loc[es_df["e"].between(-5, 16)]
    ax.plot(s["e"], s["att"], "o-", color=BLUE, lw=2.5, label="weighted (B6)")
for (name, path), color in zip(alt_paths.items(), [ORANGE, AQUA, YELLOW]):
    if path is not None:
        s = path.loc[path["e"].between(-5, 16)]
        ax.plot(s["e"], s["att"], "o-", color=color, ms=4, label=name)
ax.axhline(0, color=AXIS, lw=1); ax.axvline(-0.5, color=AXIS, lw=1)
ax.set_xlabel("years since reform"); ax.set_ylabel("divorces per 1,000 people")
ax.set_title("Same design, different weights and samples"); ax.legend(fontsize=8); plt.show()

1. Describe how each alternative differs from the weighted path, and identify which single state drives each
   difference.
2. Nevada's divorce rate in 1959 was about 34 per 1,000, roughly fifteen times the national rate. Why? What does that imply
   about **SUTVA** (no interference between units) for Nevada, and about where a divorce is *counted*?
3. Which parameter would you report, per-person or per-state, and with or without Nevada? Justify the choice by the
   question, not by the result.

**Your answer (B7):**

1. 
2. 
3. 

---
### B8 · What counts as "unilateral"? (3 points)

Researchers disagree about when (and whether) some states adopted unilateral divorce. The data include several codings.
The loop below rebuilds `G` under each coding on the same states and years. Fill in the two dynamic aggregations,
$e \in [0,7]$ and $e \in [8,15]$, for each coding (`boot=False` is fine).

In [ ]:
def timing_from(col):
    yr = panel[col].astype(float)
    if col == "lfdivlaw":
        yr = yr.where(yr != 2000, 0)
    # adopters need at least one pre-period in 1959–1988; reforms before 1960 count as always treated (-1)
    return np.select([yr.between(1960, 1988), (yr == 0) | (yr > 1988)], [yr, 0], -1).astype(int)

coding_rows = {}
for col, who in [("lfdivlaw", "Friedberg"), ("gruber", "Gruber"), ("johnson", "Johnson & Mazingo"), ("mechoulan", "Mechoulan")]:
    data = panel.assign(G=timing_from(col))
    data = data.loc[data["G"] >= 0]
    n_never, n_adopt = (data.drop_duplicates("st")["G"] == 0).sum(), (data.drop_duplicates("st")["G"] > 0).sum()
    out = md.att_gt(data=data, yname="div_rate", tname="year", idname="id", gname="G", weightsname="weights",
                    control_group="nevertreated", base_period="universal", boot=False)
    # TODO B8: early = average of e = 0..7, late = average of e = 8..15 (md.aggte with min_e / max_e)
    early, late = None, None
    coding_rows[who] = {"never-treated states": n_never, "adopting states": n_adopt,
                        "e = 0..7": None if early is None else early.overall_att,
                        "e = 8..15": None if late is None else late.overall_att}
pd.DataFrame(coding_rows).T

Compare the codings. Which agree, and which does not? In the one that does not, what changed: the treatment, the
comparison group, or both? What should a paper do about disagreements like this?

**Your answer (B8):**

*Write your answer here.*

---
### B9 · What did we learn? (3 points)

1. What do these results say about the Coasean prediction?
2. Give two economic mechanisms that could make the effect on the divorce **rate** turn *negative* after a decade.
   Remember what the outcome measures: divorces per 1,000 **people**.
3. Which assumption must hold for the $e = 15$ estimate to be causal, and why is it harder to believe than the one
   behind $e = 0$?

**Your answer (B9):**

1. 
2. 
3. 

---
# Part C · Synthesis (10 points)

Answer both in prose, about a paragraph each.

**C1 (5 points).** For each paper, state the identifying assumption as a sentence about *students* or *states*, not as an
equation. Then name the assumption **most likely to fail** in that application and say what you would observe in the
data if it did. (This is the format of your reading notes.)

**C2 (5 points).** In both papers, a reasonable published specification was qualified by the modern toolkit. For each,
name the **single analysis choice** that mattered most for the headline, and say what you would put in the paper (a
table, a figure, a sentence) so that a reader can see that choice and judge it.

**Your answer (C1):**

*Write your answer here.*

**Your answer (C2):**

*Write your answer here.*

---
### Submission checklist
- [ ] Every `# TODO` cell is filled in and the notebook runs top to bottom (*Kernel → Restart & Run All*).
- [ ] Every **Your answer** cell is filled in, with numbers from your output.
- [ ] Upload the executed `.ipynb` to Canvas by **Friday 6 November, 11:59 PM**.